# Tree species classification: pilot experiment

This notebook runs a **pilot (small-sample) version** of the study from Section 5 of the article and produces every number, table and figure needed for the Results section.

**How to run**

1. `Runtime` → `Change runtime type` → choose **T4 GPU** → `Save`.
2. `Runtime` → `Run all`.
3. When Google asks for access to Google Drive, allow it. Progress is saved to Drive, so if Colab disconnects you can simply press `Run all` again and it continues where it stopped.
4. Wait about 1.5-2 hours. Keep this browser tab open.
5. At the end the file **`pilot_results.zip`** is downloaded automatically (it is also saved in Google Drive, folder `tree_pilot`). Send this file back.

Nothing needs to be edited.

In [ ]:
# ---------------- 1. Settings ----------------
USE_DRIVE = True            # save progress to Google Drive (recommended)

# Pilot size
N_SOURCE_OBS = 200          # observations per species from Europe + North America
IMAGES_PER_OBS = 2          # images kept per observation
MAX_CA_OBS = 300            # max Central Asian observations per species and season
MIN_LEAFON_OBS = 15         # species needs this many Central Asian leaf-on observations to be in the leaf-on test
MIN_LEAFOFF_OBS = 5         # same for the leaf-off test
SEEDS = [0, 1, 2]
MAX_EPOCHS = 15
PATIENCE = 4
BATCH_SIZE = 32
SPLIT_SEED = 42
BOOTSTRAP_SEED = 2026
N_BOOTSTRAP = 1000

MODELS = {   # name: (timm model id, learning rate)
    "M1": ("tf_efficientnetv2_s.in21k_ft_in1k", 2e-4),
    "M2": ("vit_base_patch16_224.augreg_in21k", 5e-5),
}
ABLATION_MODEL = "M1"       # retrained with an image-level split (data leakage ablation)

In [ ]:
# ---------------- 2. Environment ----------------
import os, sys, json, time, io, math, random, platform, subprocess, zipfile, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)

import numpy as np
import pandas as pd
import requests
import torch
import timm
import sklearn
from PIL import Image

if not torch.cuda.is_available():
    print("WARNING: no GPU found. Go to Runtime -> Change runtime type -> T4 GPU, then Run all again.")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/tree_pilot")
else:
    OUT = Path(os.environ.get("PILOT_OUT", "tree_pilot"))
for sub in ["data", "runs", "tables", "figures"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

ENV = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "timm": timm.__version__,
    "scikit_learn": sklearn.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)",
    "date": time.strftime("%Y-%m-%d"),
}
json.dump(ENV, open(OUT / "environment.json", "w"), indent=2)
print(ENV)
print("Saving everything to", OUT.resolve())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'python': '3.13.16', 'torch': '2.11.0+cu130', 'timm': '1.0.30', 'scikit_learn': '1.6.1', 'numpy': '2.1.3', 'pandas': '2.2.3', 'gpu': 'Tesla T4', 'date': '2026-10-28'}
Saving everything to /content/drive/MyDrive/tree_pilot


In [ ]:
# ---------------- 3. Species list ----------------
SPECIES = [  # scientific name, genus, family, leaf habit
    ("Acer negundo", "Acer", "Sapindaceae", "deciduous"),
    ("Acer platanoides", "Acer", "Sapindaceae", "deciduous"),
    ("Betula pendula", "Betula", "Betulaceae", "deciduous"),
    ("Fraxinus pennsylvanica", "Fraxinus", "Oleaceae", "deciduous"),
    ("Larix sibirica", "Larix", "Pinaceae", "deciduous"),
    ("Picea pungens", "Picea", "Pinaceae", "evergreen"),
    ("Pinus sylvestris", "Pinus", "Pinaceae", "evergreen"),
    ("Populus nigra", "Populus", "Salicaceae", "deciduous"),
    ("Populus tremula", "Populus", "Salicaceae", "deciduous"),
    ("Prunus padus", "Prunus", "Rosaceae", "deciduous"),
    ("Prunus virginiana", "Prunus", "Rosaceae", "deciduous"),
    ("Salix alba", "Salix", "Salicaceae", "deciduous"),
    ("Sorbus aucuparia", "Sorbus", "Rosaceae", "deciduous"),
    ("Ulmus laevis", "Ulmus", "Ulmaceae", "deciduous"),
    ("Ulmus pumila", "Ulmus", "Ulmaceae", "deciduous"),
]
# order used everywhere (tables, confusion matrices): family, then genus, then species
SPECIES = sorted(SPECIES, key=lambda s: (s[2], s[1], s[0]))
NAMES = [s[0] for s in SPECIES]
IDX = {n: i for i, n in enumerate(NAMES)}
GENUS = np.array([s[1] for s in SPECIES])
FAMILY = np.array([s[2] for s in SPECIES])
HABIT = {s[0]: s[3] for s in SPECIES}
K = len(NAMES)

INAT_DATASET = "50c9509d-22c7-4a22-a47d-8c48425ef4a7"   # iNaturalist research-grade observations on GBIF
CENTRAL_ASIA = ["KZ", "KG", "UZ", "TJ", "TM"]
SEASONS = {"leafon": ["6,9"], "leafoff": ["12", "1,2"]}  # GBIF month ranges

In [ ]:
# ---------------- 4. Download observation metadata from GBIF ----------------
API = "https://api.gbif.org/v1"
HEADERS = {"User-Agent": "AITU student research project (tree species pilot)"}
REMOVED = {}   # cleaning log: reason -> count

def gbif_get(path, params, tries=5):
    for t in range(tries):
        try:
            r = requests.get(f"{API}/{path}", params=params, headers=HEADERS, timeout=60)
            if r.status_code == 200:
                return r.json()
        except requests.RequestException:
            pass
        time.sleep(2 * (t + 1))
    raise RuntimeError(f"GBIF request failed: {path} {params}")

def taxon_key(name):
    m = gbif_get("species/match", {"name": name, "kingdom": "Plantae"})
    key = m.get("acceptedUsageKey") or m.get("usageKey")
    if key is None:
        raise RuntimeError(f"Species not found on GBIF: {name}")
    return key

def fetch(name, key, extra, limit):
    out, offset = [], 0
    while len(out) < limit:
        page = gbif_get("occurrence/search", {"datasetKey": INAT_DATASET, "taxonKey": key,
                        "mediaType": "StillImage", "limit": 300, "offset": offset, **extra})
        out += page["results"]
        if page["endOfRecords"] or not page["results"]:
            break
        offset += 300
        time.sleep(0.2)
    return out[:limit]

def to_rows(recs, species, part):
    rows = []
    for rec in recs:
        if rec.get("decimalLatitude") is None or rec.get("decimalLongitude") is None:
            REMOVED["no coordinates"] = REMOVED.get("no coordinates", 0) + 1
            continue
        if not rec.get("month"):
            REMOVED["no month"] = REMOVED.get("no month", 0) + 1
            continue
        def is_open(m):  # licence of the photo, or of the observation if the photo has none
            lic = str(m.get("license") or rec.get("license") or "").lower()
            return any(x in lic for x in ("creativecommons", "publicdomain", "cc0", "cc_by", "cc-by"))
        media = [m for m in rec.get("media", []) if m.get("type") == "StillImage" and m.get("identifier") and is_open(m)]
        if not media:
            REMOVED["no openly licensed image"] = REMOVED.get("no openly licensed image", 0) + 1
            continue
        for i, m in enumerate(media[:IMAGES_PER_OBS]):
            rows.append({"obs_id": str(rec["key"]), "image_id": f'{rec["key"]}_{i}',
                         "url": m["identifier"].replace("/original.", "/medium."),
                         "species": species, "observer": str(rec.get("recordedBy", "unknown")),
                         "country": rec.get("countryCode"), "lat": rec["decimalLatitude"],
                         "lon": rec["decimalLongitude"], "month": int(rec["month"]),
                         "year": rec.get("year"), "part": part})
    return rows

META = OUT / "data" / "metadata.csv"
if META.exists():
    meta = pd.read_csv(META, dtype={"obs_id": str, "image_id": str})
    REMOVED = json.load(open(OUT / "data" / "removed_metadata.json"))
    print("Loaded saved metadata:", len(meta), "images")
else:
    rows = []
    rng = random.Random(SPLIT_SEED)
    for name in NAMES:
        key = taxon_key(name)
        eu = fetch(name, key, {"continent": "EUROPE"}, N_SOURCE_OBS)
        na = fetch(name, key, {"continent": "NORTH_AMERICA"}, N_SOURCE_OBS)
        half = N_SOURCE_OBS // 2
        take_eu = min(len(eu), max(half, N_SOURCE_OBS - len(na)))
        take_na = min(len(na), N_SOURCE_OBS - take_eu)
        src = rng.sample(eu, take_eu) + rng.sample(na, take_na)
        rows += to_rows(src, name, "source")
        n_ca = {}
        for season, ranges in SEASONS.items():
            recs = []
            for cc in CENTRAL_ASIA:
                for mr in ranges:
                    recs += fetch(name, key, {"country": cc, "month": mr}, MAX_CA_OBS)
            recs = recs[:MAX_CA_OBS]
            rows += to_rows(recs, name, season)
            n_ca[season] = len(recs)
        print(f"{name:24s} source {len(src):4d} obs | Central Asia leaf-on {n_ca['leafon']:3d}, leaf-off {n_ca['leafoff']:3d}")
    meta = pd.DataFrame(rows)
    before = len(meta)
    meta = meta.drop_duplicates("url")      # same photo listed twice
    REMOVED["duplicate photo"] = before - len(meta)
    meta.to_csv(META, index=False)
    json.dump(REMOVED, open(OUT / "data" / "removed_metadata.json", "w"), indent=2)
print(meta.groupby(["part"]).agg(images=("image_id", "count"), observations=("obs_id", "nunique")))

Loaded saved metadata: 6577 images
         images  observations
part                         
leafoff     142           115
leafon     2028          1526
source     4407          3000


In [ ]:
# ---------------- 5. Download images (resized, stored in one file) ----------------
from concurrent.futures import ThreadPoolExecutor
import pickle

IMG_STORE = OUT / "data" / "images.pkl"
SHORT_SIDE = 256

def download(url):
    for t in range(3):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            if r.status_code == 200:
                img = Image.open(io.BytesIO(r.content)).convert("RGB")
                w, h = img.size
                s = SHORT_SIDE / min(w, h)
                img = img.resize((max(SHORT_SIDE, round(w * s)), max(SHORT_SIDE, round(h * s))), Image.BICUBIC)
                buf = io.BytesIO()
                img.save(buf, format="JPEG", quality=90)
                return buf.getvalue()
        except Exception:
            pass
        time.sleep(1 + t)
    return None

store = pickle.load(open(IMG_STORE, "rb")) if IMG_STORE.exists() else {}
todo = meta[~meta.image_id.isin(store.keys())]
if len(todo):
    print(f"Downloading {len(todo)} images ...")
    t0 = time.time()
    with ThreadPoolExecutor(16) as ex:
        for i, (iid, data) in enumerate(zip(todo.image_id, ex.map(download, todo.url))):
            if data is not None:
                store[iid] = data
            if (i + 1) % 1000 == 0:
                print(f"  {i + 1}/{len(todo)}  ({time.time() - t0:.0f} s)")
                pickle.dump(store, open(IMG_STORE, "wb"))
    pickle.dump(store, open(IMG_STORE, "wb"))
failed = (~meta.image_id.isin(store.keys())).sum()
REMOVED["image download failed"] = int(failed)
meta = meta[meta.image_id.isin(store.keys())].reset_index(drop=True)
print(f"{len(meta)} images available, {failed} failed downloads")

6577 images available, 0 failed downloads


In [ ]:
# ---------------- 6. Splits ----------------
from sklearn.neighbors import BallTree

def tree_groups(obs, radius_m=100):
    # observations of the same species by the same observer within 100 m are treated as one tree
    obs = obs.reset_index(drop=True)
    parent = list(range(len(obs)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    for _, idx in obs.groupby(["species", "observer"]).indices.items():
        if len(idx) < 2:
            continue
        coords = np.radians(obs.loc[idx, ["lat", "lon"]].to_numpy(float))
        for i, nb in zip(idx, BallTree(coords, metric="haversine").query_radius(coords, r=radius_m / 6371000.0)):
            for j in nb:
                parent[find(i)] = find(idx[j])
    return np.array([find(i) for i in range(len(obs))])

src = meta[meta.part == "source"].copy()
obs = src.drop_duplicates("obs_id")[["obs_id", "species", "observer", "lat", "lon"]].reset_index(drop=True)
obs["group"] = tree_groups(obs)
rng = np.random.default_rng(SPLIT_SEED)
split_of = {}
for _, g in obs.groupby("species"):
    groups = g.group.unique()
    rng.shuffle(groups)
    sizes = g.groupby("group").size().loc[groups].to_numpy()
    cum = np.cumsum(sizes) / sizes.sum()
    for grp, c in zip(groups, cum):
        split_of[grp] = "train" if c <= 0.70 else ("val" if c <= 0.85 else "test_id")
obs["split"] = obs.group.map(split_of)
src = src.merge(obs[["obs_id", "group", "split"]], on="obs_id")
n_groups_merged = len(obs) - obs.group.nunique()

# image-level split for the leakage ablation
from sklearn.model_selection import train_test_split
src_img = src.drop(columns="split").copy()
tr, rest = train_test_split(src_img.index, test_size=0.30, stratify=src_img.species, random_state=SPLIT_SEED)
va, te = train_test_split(rest, test_size=0.50, stratify=src_img.loc[rest, "species"], random_state=SPLIT_SEED)
src_img.loc[tr, "split"], src_img.loc[va, "split"], src_img.loc[te, "split"] = "train", "val", "test_id"

# Central Asian test sets; species below the minimum are removed from that set
ca = meta[meta.part != "source"].copy()
ca["split"] = ca.part.map({"leafon": "test_leafon", "leafoff": "test_leafoff"})
ca["group"] = -1
removed_species = []
for s, minimum in (("test_leafon", MIN_LEAFON_OBS), ("test_leafoff", MIN_LEAFOFF_OBS)):
    counts = ca[ca.split == s].groupby("species").obs_id.nunique().reindex(NAMES, fill_value=0)
    for sp, n in counts.items():
        if n < minimum:
            if n > 0:
                removed_species.append({"set": s, "species": sp, "observations": int(n)})
            ca = ca[~((ca.split == s) & (ca.species == sp))]

DATA = {"grouped": pd.concat([src, ca], ignore_index=True), "image_level": pd.concat([src_img, ca], ignore_index=True)}
DATA["grouped"].to_csv(OUT / "data" / "split_grouped.csv", index=False)
pd.DataFrame(removed_species, columns=["set", "species", "observations"]).to_csv(OUT / "tables" / "removed_species_from_test_sets.csv", index=False)
TEST_SETS = [s for s in ["test_id", "test_leafon", "test_leafoff"] if (DATA["grouped"].split == s).any()]
summary = DATA["grouped"].groupby("split").agg(images=("image_id", "count"), observations=("obs_id", "nunique"), species=("species", "nunique"))
print(summary)
print("Observations merged into same-tree groups:", n_groups_merged)
print("Species removed from test sets (too few observations):", removed_species)

              images  observations  species
split                                      
test_id          646           450       15
test_leafoff     128           103        4
test_leafon     2012          1512       12
train           3097          2098       15
val              664           452       15
Observations merged into same-tree groups: 170
Species removed from test sets (too few observations): [{'set': 'test_leafon', 'species': 'Fraxinus pennsylvanica', 'observations': 12}, {'set': 'test_leafon', 'species': 'Prunus virginiana', 'observations': 2}, {'set': 'test_leafoff', 'species': 'Larix sibirica', 'observations': 1}, {'set': 'test_leafoff', 'species': 'Prunus padus', 'observations': 1}, {'set': 'test_leafoff', 'species': 'Sorbus aucuparia', 'observations': 1}, {'set': 'test_leafoff', 'species': 'Populus nigra', 'observations': 4}, {'set': 'test_leafoff', 'species': 'Populus tremula', 'observations': 2}, {'set': 'test_leafoff', 'species': 'Salix alba', 'observations': 2},

In [ ]:
# ---------------- 7. Training utilities ----------------
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.metrics import f1_score

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
TRAIN_TF = T.Compose([T.RandomResizedCrop(224, scale=(0.5, 1.0)), T.RandomHorizontalFlip(), T.RandomRotation(10),
                      T.ColorJitter(0.2, 0.2, 0.2, 0.02), T.ToTensor(), T.Normalize(MEAN, STD)])
EVAL_TF = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)])
_keep = set(meta.image_id)
DECODED = {iid: np.array(Image.open(io.BytesIO(b)).convert("RGB")) for iid, b in store.items() if iid in _keep}

class Images(Dataset):
    def __init__(self, df, tf):
        self.ids, self.y, self.tf = df.image_id.tolist(), df.species.map(IDX).to_numpy(), tf
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        return self.tf(Image.fromarray(DECODED[self.ids[i]])), self.y[i]

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def loader(df, tf, shuffle=False, seed=0):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(Images(df, tf), batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=shuffle,
                      num_workers=2, generator=g, pin_memory=DEVICE == "cuda")

@torch.no_grad()
def predict(model, df):
    model.eval()
    out = []
    for x, _ in loader(df, EVAL_TF):
        with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
            out.append(torch.softmax(model(x.to(DEVICE)).float(), 1).cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def features(model, df):
    model.eval()
    out = []
    for x, _ in loader(df, EVAL_TF):
        with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
            f = model.forward_head(model.forward_features(x.to(DEVICE)), pre_logits=True)
        out.append(f.float().cpu().numpy())
    return np.concatenate(out)

def save_preds(df, probs, path):
    out = df[["image_id", "obs_id", "species", "split", "month"]].copy()
    out[[f"p{k}" for k in range(K)]] = probs
    out.to_csv(path, index=False)

def train_run(model_name, seed, split_name):
    run = f"{model_name}_seed{seed}_{split_name}"
    pred_file = OUT / "runs" / f"preds_{run}.csv"
    if pred_file.exists():
        print(run, "already done, skipped")
        return
    set_seed(seed)
    df = DATA[split_name]
    tr, va = df[df.split == "train"], df[df.split == "val"]
    counts = tr.species.value_counts().reindex(NAMES, fill_value=0).to_numpy(float)
    w = torch.tensor(np.where(counts > 0, counts.sum() / (K * np.maximum(counts, 1)), 0), dtype=torch.float32, device=DEVICE)
    model_id, lr = MODELS[model_name]
    model = timm.create_model(model_id, pretrained=PRETRAINED, num_classes=K).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
    loss_fn = nn.CrossEntropyLoss(weight=w)
    dl = loader(tr, TRAIN_TF, shuffle=True, seed=seed)
    y_val = va.species.map(IDX).to_numpy()
    best, wait, t0, history = -1, 0, time.time(), []
    ckpt = Path("/tmp") / f"{run}.pt"
    for epoch in range(MAX_EPOCHS):
        model.train()
        for x, y in dl:
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
                loss = loss_fn(model(x.to(DEVICE)), y.to(DEVICE))
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        sched.step()
        f1 = f1_score(y_val, predict(model, va).argmax(1), average="macro")
        history.append(round(float(f1), 4))
        print(f"{run} epoch {epoch + 1}: validation macro-F1 = {f1:.3f}")
        if f1 > best:
            best, wait = f1, 0
            torch.save(model.state_dict(), ckpt)
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    ev = df[df.split != "train"]
    save_preds(ev, predict(model, ev), pred_file)
    if model_name == "M1" and split_name == "grouped":   # features for M3
        np.savez_compressed(OUT / "runs" / f"features_{run}.npz",
                            **{s: features(model, df[df.split == s]) for s in df.split.unique()})
    log = {"run": run, "best_val_macro_f1": round(float(best), 4), "epochs": len(history),
           "minutes": round((time.time() - t0) / 60, 1), "lr": lr, "history": history}
    with open(OUT / "runs" / "runs_log.jsonl", "a") as f:
        f.write(json.dumps(log) + "\n")
    del model
    torch.cuda.empty_cache()

PRETRAINED = os.environ.get("PILOT_PRETRAINED", "1") == "1"

In [ ]:
# ---------------- 8. Train M1 and M2 (3 seeds each) and the image-level ablation ----------------
T_START = time.time()
for seed in SEEDS:
    for m in MODELS:
        train_run(m, seed, "grouped")
for seed in SEEDS:
    train_run(ABLATION_MODEL, seed, "image_level")
print(f"Training finished in {(time.time() - T_START) / 60:.1f} minutes")

M1_seed0_grouped already done, skipped
M2_seed0_grouped already done, skipped
M1_seed1_grouped already done, skipped
M2_seed1_grouped already done, skipped
M1_seed2_grouped already done, skipped
M2_seed2_grouped epoch 1: validation macro-F1 = 0.673
M2_seed2_grouped epoch 2: validation macro-F1 = 0.738
M2_seed2_grouped epoch 3: validation macro-F1 = 0.739
M2_seed2_grouped epoch 4: validation macro-F1 = 0.743
M2_seed2_grouped epoch 5: validation macro-F1 = 0.747
M2_seed2_grouped epoch 6: validation macro-F1 = 0.752
M2_seed2_grouped epoch 7: validation macro-F1 = 0.776
M2_seed2_grouped epoch 8: validation macro-F1 = 0.787
M2_seed2_grouped epoch 9: validation macro-F1 = 0.779
M2_seed2_grouped epoch 10: validation macro-F1 = 0.789
M2_seed2_grouped epoch 11: validation macro-F1 = 0.794
M2_seed2_grouped epoch 12: validation macro-F1 = 0.789
M2_seed2_grouped epoch 13: validation macro-F1 = 0.795
M2_seed2_grouped epoch 14: validation macro-F1 = 0.794
M2_seed2_grouped epoch 15: validation macro-

M1_seed0_image_level epoch 1: validation macro-F1 = 0.488
M1_seed0_image_level epoch 2: validation macro-F1 = 0.573
M1_seed0_image_level epoch 3: validation macro-F1 = 0.635
M1_seed0_image_level epoch 4: validation macro-F1 = 0.678
M1_seed0_image_level epoch 5: validation macro-F1 = 0.673
M1_seed0_image_level epoch 6: validation macro-F1 = 0.668
M1_seed0_image_level epoch 7: validation macro-F1 = 0.692
M1_seed0_image_level epoch 8: validation macro-F1 = 0.704
M1_seed0_image_level epoch 9: validation macro-F1 = 0.720
M1_seed0_image_level epoch 10: validation macro-F1 = 0.712
M1_seed0_image_level epoch 11: validation macro-F1 = 0.698
M1_seed0_image_level epoch 12: validation macro-F1 = 0.723
M1_seed0_image_level epoch 13: validation macro-F1 = 0.707
M1_seed0_image_level epoch 14: validation macro-F1 = 0.716
M1_seed0_image_level epoch 15: validation macro-F1 = 0.714
M1_seed1_image_level epoch 1: validation macro-F1 = 0.493
M1_seed1_image_level epoch 2: validation macro-F1 = 0.622
M1_seed1

In [ ]:
# ---------------- 9. M3: features of M1 + SVM or k-NN ----------------
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = DATA["grouped"]
for seed in SEEDS:
    pred_file = OUT / "runs" / f"preds_M3_seed{seed}_grouped.csv"
    if pred_file.exists():
        continue
    t0 = time.time()
    F = np.load(OUT / "runs" / f"features_M1_seed{seed}_grouped.npz")
    y = {s: df[df.split == s].species.map(IDX).to_numpy() for s in df.split.unique()}
    cands = [(f"SVM C={c}", make_pipeline(StandardScaler(), SVC(C=c, probability=True, random_state=seed))) for c in (0.1, 1, 10)]
    cands += [(f"k-NN k={k}", make_pipeline(StandardScaler(), KNeighborsClassifier(k, weights="distance"))) for k in (5, 15, 31)]
    best = None
    for name, clf in cands:
        clf.fit(F["train"], y["train"])
        f1 = f1_score(y["val"], clf.predict(F["val"]), average="macro")
        if best is None or f1 > best[0]:
            best = (f1, name, clf)
    f1, name, clf = best
    ev = df[df.split != "train"]
    probs = np.zeros((len(ev), K))
    for s in ev.split.unique():
        rows = np.where((ev.split == s).to_numpy())[0]
        probs[rows[:, None], clf.classes_[None, :]] = clf.predict_proba(F[s])
    save_preds(ev, probs, pred_file)
    with open(OUT / "runs" / "runs_log.jsonl", "a") as f:
        f.write(json.dumps({"run": f"M3_seed{seed}_grouped", "best_val_macro_f1": round(float(f1), 4),
                            "classifier": name, "minutes": round((time.time() - t0) / 60, 1)}) + "\n")
    print(f"M3 seed {seed}: chose {name}, validation macro-F1 {f1:.3f}")

M3 seed 0: chose SVM C=1, validation macro-F1 0.733
M3 seed 1: chose SVM C=10, validation macro-F1 0.745
M3 seed 2: chose SVM C=1, validation macro-F1 0.761


In [ ]:
# ---------------- 10. Evaluation ----------------
from scipy.stats import binomtest, chi2

ALL_MODELS = ["M1", "M2", "M3"]
SET_LABEL = {"test_id": "In-distribution", "test_leafon": "Central Asia, leaf-on", "test_leafoff": "Central Asia, leaf-off",
             "test_id_leafon": "In-distribution, Jun-Sep"}
PCOLS = [f"p{k}" for k in range(K)]

def load(model, seed, split_name="grouped"):
    p = OUT / "runs" / f"preds_{model}_seed{seed}_{split_name}.csv"
    d = pd.read_csv(p, dtype={"obs_id": str, "image_id": str})
    leafon_id = d[(d.split == "test_id") & d.month.between(6, 9)].copy()
    leafon_id["split"] = "test_id_leafon"
    return pd.concat([d, leafon_id], ignore_index=True)

def metrics(d):
    if len(d) == 0:
        return {}
    p = d[PCOLS].to_numpy()
    y = d.species.map(IDX).to_numpy()
    pred = p.argmax(1)
    top5 = np.argsort(-p, 1)[:, :5]
    wrong = pred != y
    sg, sf = GENUS[pred] == GENUS[y], FAMILY[pred] == FAMILY[y]
    ne = max(int(wrong.sum()), 1)
    return {"n": len(y), "top1": 100 * (~wrong).mean(), "top5": 100 * (top5 == y[:, None]).any(1).mean(),
            "macro_f1": f1_score(y, pred, average="macro", labels=np.unique(y)),
            "genus_acc": 100 * sg.mean(), "family_acc": 100 * sf.mean(), "n_errors": int(wrong.sum()),
            "err_intra_genus": 100 * (wrong & sg).sum() / ne, "err_intra_family": 100 * (wrong & ~sg & sf).sum() / ne,
            "err_inter_family": 100 * (wrong & ~sf).sum() / ne}

def to_obs(d):
    return d.groupby(["split", "obs_id", "species"], as_index=False)[PCOLS].mean()

def mcnemar(a, b):
    b_, c_ = int((a & ~b).sum()), int((~a & b).sum())
    if b_ + c_ == 0:
        return b_, c_, float("nan"), 1.0
    if b_ + c_ < 25:
        return b_, c_, float("nan"), float(binomtest(b_, b_ + c_, 0.5).pvalue)
    stat = (abs(b_ - c_) - 1) ** 2 / (b_ + c_)
    return b_, c_, float(stat), float(chi2.sf(stat, 1))

def boot_top1(o, rng):
    correct = (o[PCOLS].to_numpy().argmax(1) == o.species.map(IDX).to_numpy()).astype(float)
    if len(correct) == 0:
        return np.full(N_BOOTSTRAP, np.nan)
    return 100 * np.array([correct[rng.integers(0, len(correct), len(correct))].mean() for _ in range(N_BOOTSTRAP)])

runs = {(m, s): load(m, s) for m in ALL_MODELS for s in SEEDS}
EVAL_SETS = TEST_SETS + (["test_id_leafon"] if "test_leafon" in TEST_SETS else [])

# ---- main table: mean and SD over seeds (image level) ----
rows = []
for (m, s), d in runs.items():
    for t in ["val"] + EVAL_SETS:
        rows.append({"model": m, "seed": s, "set": t, **metrics(d[d.split == t])})
per_seed = pd.DataFrame(rows)
per_seed.round(3).to_csv(OUT / "tables" / "per_seed_metrics.csv", index=False)
cols = ["top1", "top5", "macro_f1", "genus_acc", "family_acc"]
main = per_seed[per_seed.set != "val"].groupby(["set", "model"])[cols].agg(["mean", "std"])
main.round(3).to_csv(OUT / "tables" / "main_results_mean_sd.csv")

# ---- median-seed run per model (by validation macro-F1) ----
val = per_seed[per_seed.set == "val"].set_index(["model", "seed"]).macro_f1
MED = {m: int(val[m].sort_values().index[len(SEEDS) // 2]) for m in ALL_MODELS}
OBS = {m: to_obs(runs[(m, MED[m])]) for m in ALL_MODELS}

# ---- bootstrap confidence intervals (observation level) ----
ci_rows = []
for m in ALL_MODELS:
    for t in EVAL_SETS:
        o = OBS[m][OBS[m].split == t]
        b = boot_top1(o, np.random.default_rng(BOOTSTRAP_SEED))
        ci_rows.append({"model": m, "set": t, "n_obs": len(o), "top1_obs": metrics(o).get("top1", np.nan),
                        "ci_low": np.nanpercentile(b, 2.5), "ci_high": np.nanpercentile(b, 97.5)})
ci = pd.DataFrame(ci_rows)
ci.round(2).to_csv(OUT / "tables" / "bootstrap_ci_observation_level.csv", index=False)

# ---- generalization gaps with bootstrap CIs ----
def gap_rows():
    out = []
    pairs = [("region gap (ID minus leaf-on)", "test_id", "test_leafon"),
             ("region gap, same months (ID Jun-Sep minus leaf-on)", "test_id_leafon", "test_leafon"),
             ("season gap (leaf-on minus leaf-off)", "test_leafon", "test_leafoff")]
    for m in ALL_MODELS:
        seed_gaps = per_seed[per_seed.model == m].pivot_table(index="seed", columns="set", values="top1")
        for name, a, b in pairs:
            if a not in seed_gaps or b not in seed_gaps:
                continue
            g = seed_gaps[a] - seed_gaps[b]
            ba = boot_top1(OBS[m][OBS[m].split == a], np.random.default_rng(BOOTSTRAP_SEED))
            bb = boot_top1(OBS[m][OBS[m].split == b], np.random.default_rng(BOOTSTRAP_SEED + 1))
            out.append({"model": m, "gap": name, "mean_over_seeds_pp": g.mean(), "sd_over_seeds_pp": g.std(),
                        "median_seed_obs_level_pp": metrics(OBS[m][OBS[m].split == a])["top1"] - metrics(OBS[m][OBS[m].split == b])["top1"],
                        "ci_low": np.nanpercentile(ba - bb, 2.5), "ci_high": np.nanpercentile(ba - bb, 97.5)})
    return pd.DataFrame(out)
gaps = gap_rows()
gaps.round(2).to_csv(OUT / "tables" / "generalization_gaps.csv", index=False)

# ---- McNemar tests (observation level, median-seed runs) ----
mc_rows = []
for t in TEST_SETS:
    for a, b in [("M2", "M1"), ("M3", "M1"), ("M2", "M3")]:
        oa = OBS[a][OBS[a].split == t].sort_values("obs_id")
        ob = OBS[b][OBS[b].split == t].sort_values("obs_id")
        y = oa.species.map(IDX).to_numpy()
        ca_, cb_ = oa[PCOLS].to_numpy().argmax(1) == y, ob[PCOLS].to_numpy().argmax(1) == y
        bb_, cc_, stat, p = mcnemar(ca_, cb_)
        mc_rows.append({"set": t, "pair": f"{a} vs {b}", "n_obs": len(y), "b": bb_, "c": cc_, "chi2": stat, "p": p,
                        "significant_0.05": p < 0.05, "higher": a if bb_ > cc_ else (b if cc_ > bb_ else "equal")})
mcn = pd.DataFrame(mc_rows)
mcn.round(4).to_csv(OUT / "tables" / "mcnemar.csv", index=False)

# ---- ablation: image-level vs tree-level split (M1) ----
abl_rows = []
for s in SEEDS:
    d_img = load(ABLATION_MODEL, s, "image_level")
    for t in TEST_SETS:
        abl_rows.append({"seed": s, "set": t, "top1_tree_split": metrics(runs[(ABLATION_MODEL, s)].query("split == @t"))["top1"],
                         "top1_image_split": metrics(d_img[d_img.split == t])["top1"]})
abl = pd.DataFrame(abl_rows)
abl["difference_pp"] = abl.top1_image_split - abl.top1_tree_split
abl.round(2).to_csv(OUT / "tables" / "split_ablation_per_seed.csv", index=False)
abl_sum = abl.groupby("set")[["top1_tree_split", "top1_image_split", "difference_pp"]].agg(["mean", "std"])
abl_sum.round(2).to_csv(OUT / "tables" / "split_ablation.csv")

# ---- per group (deciduous / evergreen), error structure and per species ----
grp_rows, sp_rows = [], []
for m in ALL_MODELS:
    d = runs[(m, MED[m])]
    for t in TEST_SETS:
        dt = d[d.split == t]
        habit = dt.species.map(HABIT)
        for h in ["deciduous", "evergreen"]:
            if (habit == h).any():
                grp_rows.append({"model": m, "set": t, "group": h, **metrics(dt[habit == h])})
        grp_rows.append({"model": m, "set": t, "group": "all", **metrics(dt)})
        y = dt.species.map(IDX).to_numpy()
        pred = dt[PCOLS].to_numpy().argmax(1)
        for sp in sorted(dt.species.unique(), key=IDX.get):
            k = IDX[sp]
            tp = int(((pred == k) & (y == k)).sum()); fn = int(((pred != k) & (y == k)).sum()); fp = int(((pred == k) & (y != k)).sum())
            prec = tp / (tp + fp) if tp + fp else np.nan
            rec = tp / (tp + fn) if tp + fn else np.nan
            top_wrong = pd.Series(pred[(y == k) & (pred != k)]).map(lambda j: NAMES[j]).value_counts().head(3)
            sp_rows.append({"model": m, "set": t, "species": sp, "n_images": int((y == k).sum()), "precision": prec, "recall": rec,
                            "f1": (2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0) if not np.isnan(prec + rec) else np.nan,
                            "most_common_wrong_predictions": "; ".join(f"{n} ({c})" for n, c in top_wrong.items())})
pd.DataFrame(grp_rows).round(2).to_csv(OUT / "tables" / "per_group_and_error_structure.csv", index=False)
per_species = pd.DataFrame(sp_rows)
per_species.round(3).to_csv(OUT / "tables" / "per_species.csv", index=False)

# ---- confusion matrices and confident errors ----
from sklearn.metrics import confusion_matrix
for m in ALL_MODELS:
    d = runs[(m, MED[m])]
    for t in TEST_SETS:
        dt = d[d.split == t]
        cm = confusion_matrix(dt.species.map(IDX), dt[PCOLS].to_numpy().argmax(1), labels=range(K))
        pd.DataFrame(cm, index=NAMES, columns=NAMES).to_csv(OUT / "tables" / f"confusion_{m}_{t}.csv")
conf_rows = []
for m in ALL_MODELS:
    o = OBS[m]
    for t in TEST_SETS:
        ot = o[o.split == t]
        p = ot[PCOLS].to_numpy(); pred = p.argmax(1); y = ot.species.map(IDX).to_numpy()
        mask = (pred != y) & (p.max(1) > 0.8)
        for (_, r), pr, conf in zip(ot[mask].iterrows(), pred[mask], p.max(1)[mask]):
            conf_rows.append({"model": m, "set": t, "obs_id": r.obs_id, "true": r.species, "predicted": NAMES[pr],
                              "confidence": round(float(conf), 3),
                              "error_type": "intra-genus" if GENUS[pr] == GENUS[IDX[r.species]] else
                                            ("intra-family" if FAMILY[pr] == FAMILY[IDX[r.species]] else "inter-family")})
pd.DataFrame(conf_rows, columns=["model", "set", "obs_id", "true", "predicted", "confidence", "error_type"]).to_csv(
    OUT / "tables" / "confident_errors.csv", index=False)

# ---- dataset table ----
dg = DATA["grouped"]
ds = dg.groupby("split").agg(images=("image_id", "count"), observations=("obs_id", "nunique"), species=("species", "nunique"))
cls = dg.groupby(["split", "species"]).obs_id.nunique().groupby("split").agg(["min", "max"])
ds["class_ratio"] = (cls["max"] / cls["min"]).round(1)
ds.to_csv(OUT / "tables" / "dataset_summary.csv")
dg.groupby(["species", "split"]).obs_id.nunique().unstack(fill_value=0).reindex(NAMES).to_csv(OUT / "tables" / "observations_per_species.csv")
REMOVED_ALL = {**REMOVED, "observations merged into same-tree groups": int(n_groups_merged)}
json.dump(REMOVED_ALL, open(OUT / "tables" / "removed_records.json", "w"), indent=2)
print(main.round(2))
print(mcn)
print(gaps.round(2))

                       top1         top5       macro_f1       genus_acc        \
                       mean   std   mean   std     mean   std      mean   std   
set            model                                                            
test_id        M1     72.91  2.09  94.27  0.67     0.73  0.02     77.14  1.76   
               M2     82.20  1.72  97.52  0.56     0.82  0.02     85.96  1.66   
               M3     74.87  1.61  94.94  0.63     0.75  0.02     79.31  1.35   
test_id_leafon M1     84.07  3.07  97.35  0.88     0.80  0.06     89.09  3.68   
               M2     88.50  2.34  99.71  0.51     0.84  0.05     95.28  0.51   
               M3     84.66  2.84  97.64  1.02     0.80  0.03     90.86  2.23   
test_leafoff   M1     55.47  3.58  89.58  2.51     0.58  0.02     58.07  3.25   
               M2     67.97  3.41  92.97  0.78     0.68  0.06     70.57  4.58   
               M3     63.02  2.51  91.67  1.63     0.59  0.06     64.84  2.82   
test_leafon    M1     42.16 

In [ ]:
# ---------------- 11. Figures (vector PDF) ----------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#888", "axes.linewidth": 0.6, "pdf.fonttype": 42})
COL = {"M1": "#2a78d6", "M2": "#eb6834", "M3": "#1baf7a"}
HATCH = {"M1": "", "M2": "////", "M3": "...."}
LAB = {"M1": "M1 EfficientNetV2-S", "M2": "M2 ViT-B/16", "M3": "M3 M1 features + SVM/k-NN"}
XLAB = {"test_id": "In-distribution\ntest", "test_leafon": "Central Asia\n(leaf-on)", "test_leafoff": "Central Asia\n(leaf-off)"}

# Figure A: Top-1 accuracy, mean +- SD over seeds
fig, ax = plt.subplots(figsize=(6.3, 3.0))
w = 0.25
for i, m in enumerate(ALL_MODELS):
    for j, t in enumerate(TEST_SETS):
        r = per_seed[(per_seed.model == m) & (per_seed.set == t)].top1
        x = j + (i - 1) * (w + 0.02)
        ax.bar(x, r.mean(), w, yerr=r.std(), color=COL[m], hatch=HATCH[m], edgecolor="white", linewidth=0.8,
               error_kw={"elinewidth": 0.9, "capsize": 2.5, "ecolor": "#333"})
        ax.text(x, r.mean() + r.std() + 1.2, f"{r.mean():.1f}", ha="center", va="bottom", fontsize=6.5, color="#333")
ax.set_xticks(range(len(TEST_SETS)), [XLAB[t] for t in TEST_SETS])
ax.set_ylabel("Top-1 accuracy (%)")
ax.set_ylim(0, 105)
ax.yaxis.grid(True, color="#e6e6e6", linewidth=0.6); ax.set_axisbelow(True)
ax.legend(handles=[Patch(facecolor=COL[m], hatch=HATCH[m], edgecolor="white", label=LAB[m]) for m in ALL_MODELS],
          loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=False, fontsize=7.5)
fig.tight_layout()
fig.savefig(OUT / "figures" / "fig_pilot_accuracy.pdf", bbox_inches="tight")
plt.close(fig)

# Figure B: generalization gaps with bootstrap 95% CI (median-seed runs, observation level)
gplot = gaps[gaps.gap.str.startswith("region gap, same months") | gaps.gap.str.startswith("season")]
if len(gplot):
    names = list(dict.fromkeys(gplot.gap))
    fig, ax = plt.subplots(figsize=(5.2, 2.8))
    for i, m in enumerate(ALL_MODELS):
        for j, g in enumerate(names):
            r = gplot[(gplot.model == m) & (gplot.gap == g)]
            if not len(r):
                continue
            r = r.iloc[0]
            x = j + (i - 1) * 0.22
            ax.errorbar(x, r.median_seed_obs_level_pp, yerr=[[r.median_seed_obs_level_pp - r.ci_low], [r.ci_high - r.median_seed_obs_level_pp]],
                        fmt="o", color=COL[m], markersize=6, capsize=3, elinewidth=1.2, label=LAB[m] if j == 0 else None)
    ax.axhline(10, color="#444", linestyle="--", linewidth=0.9, label="10 pp success limit")
    ax.axhline(0, color="#999", linewidth=0.6)
    ax.set_xticks(range(len(names)), ["Region gap" if n.startswith("region") else "Season gap" for n in names])
    ax.set_ylabel("Accuracy drop (pp)")
    ax.yaxis.grid(True, color="#e6e6e6", linewidth=0.6); ax.set_axisbelow(True)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False, fontsize=7.5)
    fig.tight_layout()
    fig.savefig(OUT / "figures" / "fig_pilot_gaps.pdf", bbox_inches="tight")
    plt.close(fig)

# Figure C: confusion matrices (row %), best model by mean leaf-on accuracy
best_model = per_seed[per_seed.set == ("test_leafon" if "test_leafon" in TEST_SETS else "test_id")].groupby("model").top1.mean().idxmax()
sets_c = [t for t in ["test_leafon", "test_leafoff"] if t in TEST_SETS] or ["test_id"]
fig, axes = plt.subplots(1, len(sets_c), figsize=(5.6 * len(sets_c), 5.2), layout="constrained")
axes = np.atleast_1d(axes)
cmap = plt.get_cmap("Blues").copy()
cmap.set_bad("#f2f2f2")   # grey rows: species not in this test set
for ax, t in zip(axes, sets_c):
    cm = pd.read_csv(OUT / "tables" / f"confusion_{best_model}_{t}.csv", index_col=0).to_numpy(float)
    rs = cm.sum(1, keepdims=True)
    cmr = np.where(rs > 0, 100 * cm / np.maximum(rs, 1), np.nan)
    rows_n = NAMES
    im = ax.imshow(cmr, cmap=cmap, vmin=0, vmax=100, aspect="auto")
    for i in range(cmr.shape[0]):
        for j in range(K):
            if not np.isnan(cmr[i, j]) and cmr[i, j] >= 5:
                ax.text(j, i, f"{cmr[i, j]:.0f}", ha="center", va="center", fontsize=6, color="white" if cmr[i, j] > 55 else "#222")
    fam_edges = [j - 0.5 for j in range(1, K) if FAMILY[j] != FAMILY[j - 1]]
    for e in fam_edges:
        ax.axvline(e, color="#555", linewidth=0.7)
        ax.axhline(e, color="#555", linewidth=0.7)
    ax.set_xticks(range(K), NAMES, rotation=65, ha="right", fontsize=6.5, style="italic")
    ax.set_yticks(range(len(rows_n)), rows_n, fontsize=6.5, style="italic")
    ax.set_xlabel("Predicted species"); ax.set_ylabel("True species")
    ax.set_title(f"{best_model}, {SET_LABEL[t]} (grey rows: species not in this test set)", fontsize=8, loc="left")
cb = fig.colorbar(im, ax=axes.tolist(), fraction=0.025, pad=0.02)
cb.set_label("Share of true-species images (%)")
fig.savefig(OUT / "figures" / "fig_pilot_confusion.pdf", bbox_inches="tight")
plt.close(fig)
print("Figures saved:", [p.name for p in (OUT / "figures").glob("*.pdf")])

Figures saved: ['fig_pilot_accuracy.pdf', 'fig_pilot_gaps.pdf', 'fig_pilot_confusion.pdf']


In [ ]:
# ---------------- 12. Text report and download ----------------
log = [json.loads(l) for l in open(OUT / "runs" / "runs_log.jsonl")]
total_min = sum(l.get("minutes", 0) for l in log)
lines = ["# Pilot results", "", f"Environment: {json.dumps(ENV)}", f"Total training/fitting time: {total_min:.0f} minutes", "",
         "## Dataset", ds.to_string(), "", "Removed records: " + json.dumps(REMOVED_ALL),
         "Species removed from test sets: " + json.dumps(removed_species), "",
         "## Main results (image level, mean and SD over seeds)", main.round(2).to_string(), "",
         "## Bootstrap 95% CI (observation level, median-seed run)", ci.round(2).to_string(index=False), "",
         "Median-seed runs: " + json.dumps(MED), "",
         "## McNemar tests", mcn.round(4).to_string(index=False), "",
         "## Generalization gaps", gaps.round(2).to_string(index=False), "",
         "## Split ablation (M1)", abl_sum.round(2).to_string(), "",
         "## Per group and error structure (median-seed runs)", pd.DataFrame(grp_rows).round(2).to_string(index=False), "",
         "## Per species (median-seed runs)", per_species.round(3).to_string(index=False), "",
         "## Runs", "\n".join(json.dumps(l) for l in log)]
(OUT / "report.txt").write_text("\n".join(lines))

zip_path = Path("pilot_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in ["report.txt", "environment.json"]:
        z.write(OUT / p, p)
    for sub in ["tables", "figures"]:
        for p in (OUT / sub).glob("*"):
            z.write(p, f"{sub}/{p.name}")
    z.write(OUT / "runs" / "runs_log.jsonl", "runs_log.jsonl")
    z.write(OUT / "data" / "metadata.csv", "data/metadata.csv")
shutil.copy(zip_path, OUT / "pilot_results.zip")
print("\n".join(lines[:40]))
print("\nDONE. pilot_results.zip is saved in Google Drive (tree_pilot) and is being downloaded.")
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))

# Pilot results

Environment: {"python": "3.13.16", "torch": "2.11.0+cu130", "timm": "1.0.30", "scikit_learn": "1.6.1", "numpy": "2.1.3", "pandas": "2.2.3", "gpu": "Tesla T4", "date": "2026-10-28"}
Total training/fitting time: 78 minutes

## Dataset
              images  observations  species  class_ratio
split                                                   
test_id          646           450       15          1.0
test_leafoff     128           103        4          5.5
test_leafon     2012          1512       12         18.8
train           3097          2098       15          1.0
val              664           452       15          1.0

Removed records: {"duplicate photo": 1, "image download failed": 0, "observations merged into same-tree groups": 170}
Species removed from test sets: [{"set": "test_leafon", "species": "Fraxinus pennsylvanica", "observations": 12}, {"set": "test_leafon", "species": "Prunus virginiana", "observations": 2}, {"set": "test_leafoff", "species": "Larix s

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>